In [ ]:
from pathlib import Path

def find_project_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / 'README.md').exists() and (p / 'data').exists():
            return p
    raise FileNotFoundError('Project root not found. Make sure README.md and data/ exist.')

PROJECT_ROOT = find_project_root()

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
FIGURES_DIR = PROJECT_ROOT / 'figures'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_DIR =', RAW_DIR)
print('PROCESSED_DIR =', PROCESSED_DIR)

import sys
!{sys.executable} -m pip install numpy pandas matplotlib mne

import pandas as pd

hypno_df = pd.read_csv(PROCESSED_DIR / '../data/processed/hypno_df.csv')
hypno_df.head()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne

In [ ]:
from mne.datasets.sleep_physionet.age import fetch_data

subjects = [0]
recordings = [1]

data_paths = fetch_data(subjects, recordings)

In [ ]:
raw = mne.io.read_raw_edf(data_paths[0][0], preload=True)

print(raw.info)

In [ ]:
annotations = raw.annotations
annotations_df = annotations.to_data_frame()

annotations_df.head()

In [ ]:
stage_counts = annotations_df['description'].value_counts()

stage_counts.plot(kind='bar')
plt.title("Sleep Stage Distribution")
plt.xlabel("Sleep Stage")
plt.ylabel("Count")
plt.show()

In [ ]:
total = len(annotations_df)

for stage, count in stage_counts.items():
    print(stage, ":", round(count / total * 100, 2), "%")

In [ ]:
transitions = (annotations_df['description'] != annotations_df['description'].shift()).sum()

print("Sleep fragmentation:", transitions)

In [ ]:
annotations_df['description'].unique()

In [ ]:
annotations_df.head(20)

In [ ]:
wake_transitions = ((annotations_df['description'] == 'Sleep stage W') & 
                    (annotations_df['description'].shift() != 'Sleep stage W')).sum()

print("Awakenings:", wake_transitions)

In [ ]:
print("Stage transitions:", len(stage_changes))
print("Awakenings:", wake_transitions)

In [ ]:
stages = annotations_df['description']

stage_changes = stages[stages != stages.shift()]

print("Stage transitions:", len(stage_changes))

In [ ]:
hypno = mne.read_annotations(data_paths[0][1])
hypno.to_data_frame()

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
expanded_stages = []

for _, row in hypno_df.iterrows():
    n_points = int(row['duration'] // 30)  # 30 сек эпохи Sleep-EDF
    expanded_stages.extend([row['description']] * n_points)

expanded_stages = pd.Series(expanded_stages)

In [ ]:
import mne
from mne.datasets.sleep_physionet.age import fetch_data

subjects = [0]
recordings = [1]

data_paths = fetch_data(subjects, recordings)

hypno = mne.read_annotations(data_paths[0][1])
hypno_df = hypno.to_data_frame()

hypno_df.head()

In [ ]:
transitions = (expanded_stages != expanded_stages.shift()).sum()

print("Sleep fragmentation:", transitions)

In [ ]:
import pandas as pd

expanded_stages = pd.Series(expanded_stages)

transitions = (expanded_stages != expanded_stages.shift()).sum()

print("Sleep fragmentation:", transitions)

In [ ]:
expanded_stages.value_counts(normalize=True) * 100

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,4))
plt.plot(expanded_stages.astype('category').cat.codes)
plt.title("Hypnogram (Expanded Timeline)")
plt.xlabel("Time (30s epochs)")
plt.ylabel("Sleep Stage")
plt.show()

In [ ]:
len(expanded_stages)

In [ ]:
expanded_stages = []

for _, row in hypno_df.iterrows():
    n_points = max(1, int(row['duration'] / 30))  # 👈 ВАЖНО: /
    expanded_stages.extend([row['description']] * n_points)

import pandas as pd
expanded_stages = pd.Series(expanded_stages)

print("Length:", len(expanded_stages))

In [ ]:
print(hypno_df['duration'].describe())
print(hypno_df['duration'].head(10))

In [ ]:
hypno_df['description'].value_counts()

In [ ]:
stage_duration = hypno_df.groupby('description')['duration'].sum()

stage_duration

In [ ]:
(stage_duration / stage_duration.sum()) * 100

In [ ]:
transitions = (hypno_df['description'] != hypno_df['description'].shift()).sum()

print("Sleep stage transitions:", transitions)

In [ ]:
total_duration_hours = hypno_df['duration'].sum() / 3600

fragmentation_index = 154 / total_duration_hours

print(fragmentation_index)

In [ ]:
stage_duration = hypno_df.groupby('description')['duration'].sum()

(stage_duration / stage_duration.sum()) * 100

In [ ]:
print("Total sleep time (hrs):", hypno_df['duration'].sum() / 3600)
print("Transitions:", 154)
print("Unique stages:", hypno_df['description'].nunique())

In [ ]:
df = hypno_df.copy()

df = df[~df['description'].str.contains('\?')]

In [ ]:
df = hypno_df.copy()

df = df[~df['description'].str.contains('?', regex=False, na=False)]

In [ ]:
total_time = df['duration'].sum()
print("Total sleep time (sec):", total_time)


In [ ]:
stage_time = df.groupby('description')['duration'].sum()
stage_pct = stage_time / total_time * 100


In [ ]:
features['rem_pct'] = stage_pct.get('Sleep stage R', 0)

features['deep_sleep_pct'] = (
    stage_pct.get('Sleep stage 3', 0) +
    stage_pct.get('Sleep stage 4', 0)
)

features['wake_pct'] = stage_pct.get('Sleep stage W', 0)
⚡ Sleep efficiency
features['sleep_efficiency'] = (
    (total_time - df[df['description'] == 'Sleep stage W']['duration'].sum())
    / total_time


In [ ]:
import pandas as pd

df = hypno_df.copy()

df = df[~df['description'].str.contains('?', regex=False, na=False)]

total_time = df['duration'].sum()
print("Total sleep time (sec):", total_time)

stage_time = df.groupby('description')['duration'].sum()
stage_pct = stage_time / total_time * 100

print(stage_pct)

features = {}

features['total_sleep_hours'] = total_time / 3600
features['fragmentation'] = (df['description'] != df['description'].shift()).sum()

features['rem_pct'] = stage_pct.get('Sleep stage R', 0)

features['deep_sleep_pct'] = (
    stage_pct.get('Sleep stage 3', 0) +
    stage_pct.get('Sleep stage 4', 0)
)

features['wake_pct'] = stage_pct.get('Sleep stage W', 0)

features['sleep_efficiency'] = (
    (total_time - df[df['description'] == 'Sleep stage W']['duration'].sum())
    / total_time
)

features_df = pd.DataFrame([features])

print(features_df)

In [ ]:
import pandas as pd

features_df = pd.DataFrame([features])

features_df


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = hypno_df.copy()

df = df[~df['description'].str.contains('?', regex=False, na=False)].copy()

df = df[df['duration'] > 0].copy()

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nUnique stages:")
print(df['description'].value_counts())

In [ ]:
total_time = df['duration'].sum()

stage_time = df.groupby('description')['duration'].sum().sort_values(ascending=False)
stage_hours = stage_time / 3600
stage_pct = stage_time / total_time * 100

stage_summary = pd.DataFrame({
    'duration_sec': stage_time,
    'duration_hours': stage_hours,
    'percent': stage_pct
})

stage_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(stage_summary.index, stage_summary['duration_hours'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Duration (hours)')
plt.title('Sleep Stage Distribution')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
print("First 20 rows:")
display(df[['description', 'duration']].head(20))

print("Last 20 rows:")
display(df[['description', 'duration']].tail(20))

In [ ]:
print("First 20 rows:")
display(df[['description', 'duration']].head(20))

print("Last 20 rows:")
display(df[['description', 'duration']].tail(20))

In [ ]:
stage_map = {
    'Sleep stage W': 4,
    'Sleep stage R': 3,
    'Sleep stage 1': 2,
    'Sleep stage 2': 1,
    'Sleep stage 3': 0,
    'Sleep stage 4': 0
}

stage_labels = {
    4: 'Wake',
    3: 'REM',
    2: 'N1',
    1: 'N2',
    0: 'N3'
}

plot_df = df[df['description'].isin(stage_map.keys())].copy()
plot_df['stage_num'] = plot_df['description'].map(stage_map)

x = []
y = []
current_time = 0

for _, row in plot_df.iterrows():
    start = current_time / 3600
    end = (current_time + row['duration']) / 3600
    stage = row['stage_num']
    
    x.extend([start, end])
    y.extend([stage, stage])
    
    current_time += row['duration']

plt.figure(figsize=(14, 5))
plt.step(x, y, where='post', linewidth=2)

plt.yticks(list(stage_labels.keys()), [stage_labels[k] for k in stage_labels.keys()])
plt.gca().invert_yaxis()
plt.xlabel('Time (hours)')
plt.ylabel('Sleep stage')
plt.title('Hypnogram')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
features = {}

features['total_sleep_hours'] = total_time / 3600
features['fragmentation'] = (df['description'] != df['description'].shift()).sum()

features['rem_pct'] = stage_pct.get('Sleep stage R', 0)

features['deep_sleep_pct'] = (
    stage_pct.get('Sleep stage 3', 0) +
    stage_pct.get('Sleep stage 4', 0)
)

features['wake_pct'] = stage_pct.get('Sleep stage W', 0)

wake_time = df[df['description'] == 'Sleep stage W']['duration'].sum()
features['sleep_efficiency'] = (
    (total_time - wake_time) / total_time if total_time > 0 else np.nan
)

features_df = pd.DataFrame([features])
features_df

In [ ]:
print("Sleep Exploration Summary")
print("-------------------------")
print(f"Total recording time: {features['total_sleep_hours']:.2f} h")
print(f"Fragmentation: {features['fragmentation']}")
print(f"REM %: {features['rem_pct']:.2f}")
print(f"Deep sleep %: {features['deep_sleep_pct']:.2f}")
print(f"Wake %: {features['wake_pct']:.2f}")
print(f"Sleep efficiency: {features['sleep_efficiency']:.2%}")

In [ ]:
comments = []

if features['total_sleep_hours'] > 12:
    comments.append("Recording duration is unusually long for a single night.")

if features['wake_pct'] > 30:
    comments.append("Wake percentage is high, suggesting long wake periods in the recording.")

if features['sleep_efficiency'] < 0.7:
    comments.append("Sleep efficiency is low, which may indicate that the full recording window is being used instead of the actual sleep period.")

if len(comments) == 0:
    comments.append("Basic sanity check looks reasonable.")

print("Sanity Check")
print("------------")
for c in comments:
    print("-", c)

## Exploration conclusion

The recording was explored at the stage level using sleep stage durations and a hypnogram.

Key observations:
- Sleep stage distribution was calculated.
- A hypnogram was used to visualize stage transitions across time.
- Basic sleep metrics were computed: total duration, fragmentation, REM proportion, deep sleep proportion, wake proportion, and sleep efficiency.
- The unusually high wake proportion / long total duration suggests that additional preprocessing may be needed before feature engineering.

Next step:
Refine sleep-period logic and move to `02_feature_engineering.ipynb`, where sleep metrics will be formalized into reusable features for machine learning.

In [ ]:
hypno_df = df[['description', 'duration']].copy()
hypno_df.to_csv('../data/processed/hypno_df.csv', index=False)
hypno_df.head(20)

In [ ]:
hypno_df.to_csv(PROCESSED_DIR / '../data/processed/hypno_df.csv', index=False)